# วิธีรัน Editor round (MLP + stationary features) — ตอบ Academic Editor ข้อ 3

**ขั้นตอน**
1. วาง notebook นี้ใน `My Drive/Run_4Markets` (โฟลเดอร์เดิมที่มี `data/` และ `results/pricevol/` จากการรัน Extended arm)
2. Runtime → Change runtime type → **T4 GPU** (เลือก GPU ชนิดเดียวกันตลอด)
3. Runtime → Run all — ประมาณ 5–8 ชม. ถ้าหลุดให้กด Run all ใหม่ ส่วนที่เสร็จแล้วจะถูกข้าม
   - Part A: MLP บน features เดิม (~30 นาที)
   - Part B: stationary features: frozen arm (classical + 5 networks) และ extended arm 6 สถาปัตยกรรม
4. ทดสอบเร็วก่อนได้ด้วย `QUICK = True`
5. ผลทั้งหมดบันทึกใน Google Drive ที่ `My Drive/Run_4Markets/results/` ระหว่างรัน (Colab หลุดก็ไม่หาย)
6. เสร็จแล้วจะได้ `My Drive/Run_4Markets/results_editor_round.zip` ดาวน์โหลดจาก Drive มาวางที่ `MAKE Stock\Edit\R2\` แล้วบอก Claude
   ถ้ายังไม่มี `data/` ใน Drive notebook จะให้ upload `Run_4Markets_data.zip` ครั้งเดียวแล้วแตกไฟล์ลง Drive ให้เอง


In [ ]:
# ---------------- configuration ----------------
MARKETS = ['SET50', 'SET100', 'SPX', 'CSI300']
USE_VOLUME = 'auto'      # 'auto': volume features when the market has volume; False: same 75 price-only features for all markets
INCLUDE_2026 = False     # True adds the partial 2026 year (Jan-Sep) as an extra outer fold
QUICK = False            # True = smoke test (1 seed, 2 configs, 2 epochs)
DRIVE_DIR = '/content/drive/MyDrive/Run_4Markets'   # folder that contains data/ (used if present)
LOCAL_DIR = '/content/Run_4Markets'                 # used when Drive is not mounted

FOLDS = [(y, f'{y}-01-01', f'{y}-12-31') for y in (2022, 2023, 2024, 2025)]
if INCLUDE_2026: FOLDS.append((2026, '2026-01-01', '2026-12-31'))
SEEDS = [42] if QUICK else [42, 123, 456, 789, 2025]
EPOCHS = 2 if QUICK else 20
WIN = {'LSTM': 5, 'CNN': 20, 'LSTM-CNN': 20, 'LSTM-Attention': 10, 'LSTM-CNN-Attention': 20}
ARCHS = ['LSTM', 'CNN'] if QUICK else list(WIN)
TAG = ('quick_' if QUICK else '') + ('pricevol' if USE_VOLUME == 'auto' else 'priceonly') + ('_2026' if INCLUDE_2026 else '')

In [ ]:
# ---------------- locate data: Google Drive folder, or upload Run_4Markets_data.zip ----------------
import os, shutil, zipfile, importlib.util, subprocess, sys
for pkg in ['lightgbm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
ROOT = None
IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        os.makedirs(DRIVE_DIR, exist_ok=True)   # results are always saved in Google Drive when it is mounted
        ROOT = DRIVE_DIR
    except Exception as e:
        print('WARNING: Google Drive could not be mounted; results will be lost if the runtime resets:', e)
else:
    print('Drive not used: not running in Colab')
if ROOT is None:
    ROOT = LOCAL_DIR
    os.makedirs(ROOT, exist_ok=True)
if not (os.path.isdir(os.path.join(ROOT, 'data')) or os.path.isdir(os.path.join(ROOT, 'Run_4Markets', 'data'))):
    from google.colab import files
    print('data/ not found in', ROOT, '- upload Run_4Markets_data.zip (it is extracted into this folder once)')
    for name in files.upload():
        with zipfile.ZipFile(name) as z: z.extractall(ROOT)
DATA_DIR = os.path.join(ROOT, 'data')
if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join(ROOT, 'Run_4Markets', 'data')):
    DATA_DIR = os.path.join(ROOT, 'Run_4Markets', 'data')
OUT_ROOT = os.path.join(ROOT, 'results', TAG)
print('data:', DATA_DIR, sorted(os.listdir(DATA_DIR)))
print('results:', OUT_ROOT)

In [ ]:
# ---------------- verify files against the manifest ----------------
import hashlib, json
man = json.load(open(os.path.join(DATA_DIR, 'manifest.json')))
for f in man['files']:
    p = os.path.join(DATA_DIR, f['name'])
    ok = os.path.exists(p) and hashlib.sha256(open(p, 'rb').read()).hexdigest() == f['sha256']
    print(('OK      ' if ok else 'CHANGED ') + f['name'])

In [ ]:
# ---------------- data alignment and features ----------------
import numpy as np, pandas as pd
COLS = ['Close', 'Open', 'High', 'Low', 'Volume', 'Change_pct']
LAST_TEST = max(e for _, _, e in FOLDS)


def load(name):
    return pd.read_csv(os.path.join(DATA_DIR, name), parse_dates=['Date']).sort_values('Date').reset_index(drop=True)


def build_aligned(mk):
    """Daily rows + previous completed week (weeks start Sunday) + previous completed month; target = next session's close."""
    d, w, m = load(f'{mk}_days.csv'), load(f'{mk}_weeks.csv'), load(f'{mk}_months.csv')
    d['Target_Next_Close'] = d.Close.shift(-1)
    d = d[d.Date <= LAST_TEST]
    D = d.rename(columns={c: c + '_D' for c in COLS})
    D['wkey'] = D.Date - pd.to_timedelta((D.Date.dt.dayofweek + 1) % 7, unit='D') - pd.Timedelta(days=7)
    W = w[['Date'] + COLS].rename(columns={c: c + '_W' for c in COLS}).rename(columns={'Date': 'wdate'})
    D = pd.merge_asof(D.sort_values('wkey'), W.sort_values('wdate'), left_on='wkey', right_on='wdate').sort_values('Date')
    D['mkey'] = D.Date.dt.to_period('M').dt.start_time - pd.offsets.MonthBegin(1)
    M = m[['Date'] + COLS].rename(columns={c: c + '_M' for c in COLS}).rename(columns={'Date': 'mdate'})
    D = pd.merge_asof(D.sort_values('mkey'), M.sort_values('mdate'), left_on='mkey', right_on='mdate').sort_values('Date')
    keep = ['Date'] + [c + s for s in ['_D', '_W', '_M'] for c in COLS] + ['Target_Next_Close']
    D = D.dropna(subset=[c for c in keep if not c.startswith('Volume')])
    assert (D.wdate + pd.Timedelta(days=7) <= D.Date).all() and (D.mdate < D.Date.dt.to_period('M').dt.start_time).all()
    return D[keep].reset_index(drop=True)


def rsi(c, n=14):
    d = c.diff(); up = d.clip(lower=0); dn = -d.clip(upper=0)
    au = up.ewm(alpha=1/n, adjust=False).mean(); ad = dn.ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100/(1 + au/ad.replace(0, np.nan))


def adx(h, l, c, n=14):
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    upm = h.diff(); dnm = -l.diff()
    pdm = np.where((upm > dnm) & (upm > 0), upm, 0.0); ndm = np.where((dnm > upm) & (dnm > 0), dnm, 0.0)
    atr = tr.ewm(alpha=1/n, adjust=False).mean()
    pdi = 100*pd.Series(pdm, c.index).ewm(alpha=1/n, adjust=False).mean()/atr
    ndi = 100*pd.Series(ndm, c.index).ewm(alpha=1/n, adjust=False).mean()/atr
    dx = 100*(pdi-ndi).abs()/(pdi+ndi)
    return dx.ewm(alpha=1/n, adjust=False).mean(), pdi, ndi, atr


def build_features(a, use_vol):
    a = a.sort_values('Date').reset_index(drop=True)
    c, o, h, l = a.Close_D, a.Open_D, a.High_D, a.Low_D
    lv = np.log(a.Volume_D) if use_vol else None
    F = pd.DataFrame(index=a.index)
    for col in ['Close_D', 'Open_D', 'High_D', 'Low_D']: F[col] = a[col]
    if use_vol: F['logVol_D'] = lv
    for k in range(1, 6):
        F[f'close_lag{k}'] = c.shift(k)
        if use_vol: F[f'logvol_lag{k}'] = lv.shift(k)
    r1 = c.pct_change()
    for k in [1, 3, 5, 10, 20, 60]: F[f'ret_{k}'] = c.pct_change(k)
    for k in [5, 10, 20, 60]:
        ma = c.rolling(k).mean(); F[f'ma_{k}'] = ma; F[f'c_ma_{k}'] = c/ma - 1
        F[f'vol_{k}'] = r1.rolling(k).std()
        F[f'mom_{k}'] = c - c.shift(k); F[f'roc_{k}'] = 100*(c/c.shift(k) - 1)
        if use_vol: F[f'vr_{k}'] = lv - lv.rolling(k).mean()
    F['ma5_ma20'] = F.ma_5/F.ma_20 - 1; F['ma20_ma60'] = F.ma_20/F.ma_60 - 1
    rng_ = (h - l).replace(0, np.nan)
    F['body'] = (c - o)/o; F['range'] = (h - l)/c
    F['upper_sh'] = (h - np.maximum(o, c))/rng_; F['lower_sh'] = (np.minimum(o, c) - l)/rng_
    F['close_pos'] = (c - l)/rng_; F['gap'] = o/c.shift() - 1
    for k in range(1, 6):
        F[f'dir_lag{k}'] = (c.shift(k-1) > c.shift(k)).astype(float); F[f'ret_lag{k}'] = r1.shift(k-1)
    ll14, hh14 = l.rolling(14).min(), h.rolling(14).max()
    F['stoch_k'] = 100*(c - ll14)/(hh14 - ll14); F['stoch_d'] = F.stoch_k.rolling(3).mean()
    F['rsi14'] = rsi(c, 14); F['rsi6'] = rsi(c, 6)
    e12, e26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    F['macd'] = e12 - e26; F['macd_sig'] = F.macd.ewm(span=9, adjust=False).mean(); F['macd_hist'] = F.macd - F.macd_sig
    F['willr'] = -100*(hh14 - c)/(hh14 - ll14)
    tp = (h + l + c)/3
    F['cci20'] = (tp - tp.rolling(20).mean())/(0.015*tp.rolling(20).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True))
    F['adx14'], F['pdi14'], F['ndi14'], atr = adx(h, l, c); F['atr14_rel'] = atr/c
    for p in ['W', 'M']:
        F[f'chg_{p}'] = a[f'Change_pct_{p}']; F[f'c_vs_close_{p}'] = c/a[f'Close_{p}'] - 1
        F[f'range_{p}'] = (a[f'High_{p}'] - a[f'Low_{p}'])/a[f'Close_{p}']
        F[f'body_{p}'] = (a[f'Close_{p}'] - a[f'Open_{p}'])/a[f'Open_{p}']
    F['chg_D'] = a.Change_pct_D
    out = pd.concat([pd.DataFrame({'Date': a.Date, 'C': c, 'C_next': a.Target_Next_Close}), F], axis=1)
    out['ret_next'] = out.C_next/out.C - 1
    out['y'] = (out.C_next > out.C).astype(int); out['zero'] = out.C_next == out.C
    out = out.replace([np.inf, -np.inf], np.nan).iloc[60:].reset_index(drop=True)
    feats = list(F.columns)
    bad = out[feats].isna().sum(); assert bad.sum() == 0, bad[bad > 0]
    return out, feats

In [ ]:
# ---------------- forecasters ----------------
import time, warnings, torch, torch.nn as nn, lightgbm as lgb
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
warnings.filterwarnings('ignore')
torch.set_num_threads(1); torch.use_deterministic_algorithms(True)


def frame(method, family, seed, year, te, pred, score, arch=None):
    f = pd.DataFrame({'method': method, 'family': family, 'seed': seed, 'year': year, 'Date': te.Date.values, 'y': te.y.values,
                      'zero': te.zero.values, 'pred': np.asarray(pred).astype(int), 'score': np.asarray(score, float)})
    if arch: f.insert(2, 'arch', arch)
    return f


def run_classical(D, FEATS, path):
    if os.path.exists(path): return
    rows = []
    for year, s, e in FOLDS:
        tr = D[D.Date < s].iloc[:-1]; te = D[(D.Date >= s) & (D.Date <= e)]
        if te.empty: continue
        Xtr, Xte = tr[FEATS].values, te[FEATS].values
        sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
        ytr = tr.y.values; maj = int(ytr.mean() >= 0.5)
        rows += [frame('Train-majority', 'Naive', 0, year, te, np.full(len(te), maj), np.full(len(te), ytr.mean())),
                 frame('Persistence', 'Naive', 0, year, te, te.dir_lag1.values, te.ret_1.values),
                 frame('Reversal', 'Naive', 0, year, te, 1 - te.dir_lag1.values, -te.ret_1.values),
                 frame('MA20-trend', 'Naive', 0, year, te, (te.c_ma_20 > 0).astype(int).values, te.c_ma_20.values)]
        p = LogisticRegression(C=1.0, max_iter=5000).fit(Ztr, ytr).predict_proba(Zte)[:, 1]
        rows.append(frame('Logistic-L2', 'Classical', 0, year, te, p > 0.5, p))
        r = Ridge(alpha=1.0).fit(Ztr, tr.ret_next.values).predict(Zte)
        rows.append(frame('Ridge-return', 'Classical', 0, year, te, r > 0, r))
        for sd in SEEDS:
            p = RandomForestClassifier(n_estimators=500, min_samples_leaf=20, max_features='sqrt', n_jobs=-1,
                                       random_state=sd).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
            rows.append(frame('Random forest', 'Classical', sd, year, te, p > 0.5, p))
            p = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=50, subsample=0.8,
                                   subsample_freq=1, colsample_bytree=0.8, random_state=sd, verbose=-1).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
            rows.append(frame('LightGBM', 'Classical', sd, year, te, p > 0.5, p))
    pd.concat(rows).to_csv(path, index=False)


class CausalConv(nn.Module):
    def __init__(s, i, o, k=3):
        super().__init__(); s.k = k; s.c = nn.Conv1d(i, o, k)
    def forward(s, x):
        x = nn.functional.pad(x.transpose(1, 2), (s.k - 1, 0)); return torch.relu(s.c(x)).transpose(1, 2)


class CausalAtt(nn.Module):
    def __init__(s, d):
        super().__init__(); s.m = nn.MultiheadAttention(d, 2, batch_first=True, kdim=d, vdim=d)
    def forward(s, x):
        T = x.shape[1]; mask = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        return s.m(x, x, x, attn_mask=mask, need_weights=False)[0]


class Net(nn.Module):
    def __init__(s, arch, nf):
        super().__init__(); s.arch = arch
        if 'LSTM' in arch: s.lstm = nn.LSTM(nf, 16, batch_first=True)
        dim = 16 if 'LSTM' in arch else nf
        if 'CNN' in arch: s.conv = CausalConv(dim, 32); dim = 32
        if 'Attention' in arch: s.att = CausalAtt(dim)
        s.head = nn.Sequential(nn.Linear(dim, 8), nn.ReLU(), nn.Linear(8, 1))
    def forward(s, x):
        if s.arch == 'LSTM': return s.head(s.lstm(x)[0][:, -1])
        if 'LSTM' in s.arch: x = s.lstm(x)[0]
        if 'CNN' in s.arch: x = s.conv(x)
        if 'Attention' in s.arch: x = s.att(x)
        return s.head(x.mean(1))


def seqs(X, idx, W):
    return np.stack([X[i - W + 1:i + 1] for i in idx]).astype(np.float32)


def run_dl(D, FEATS, arch, path, mk):
    if os.path.exists(path): return
    X_all, W, out = D[FEATS].values, WIN[arch], []
    for year, s, e in FOLDS:
        tr_idx = np.where(D.Date < s)[0][:-1]; tr_idx = tr_idx[tr_idx >= W - 1]
        te_idx = np.where((D.Date >= s) & (D.Date <= e))[0]
        if len(te_idx) == 0: continue
        sx = MinMaxScaler().fit(X_all[tr_idx]); Xs = sx.transform(X_all)
        sy = MinMaxScaler().fit(D.C_next.values[tr_idx, None])
        Xtr, Xte = seqs(Xs, tr_idx, W), seqs(Xs, te_idx, W)
        targets = {'R': sy.transform(D.C_next.values[tr_idx, None]).astype(np.float32), 'C': D.y.values[tr_idx, None].astype(np.float32)}
        te = D.iloc[te_idx]
        for obj in ['R', 'C']:
            for sd in SEEDS:
                t0 = time.time(); torch.manual_seed(sd); np.random.seed(sd)
                net = Net(arch, X_all.shape[1]); opt = torch.optim.Adam(net.parameters(), 1e-3)
                lossf = nn.MSELoss() if obj == 'R' else nn.BCEWithLogitsLoss()
                xt, yt = torch.from_numpy(Xtr), torch.from_numpy(targets[obj])
                net.train()
                for ep in range(EPOCHS):
                    for b in range(0, len(xt), 32):
                        opt.zero_grad(); loss = lossf(net(xt[b:b+32]), yt[b:b+32]); loss.backward(); opt.step()
                net.eval()
                with torch.no_grad(): o = net(torch.from_numpy(Xte)).numpy().ravel()
                if obj == 'R': score = sy.inverse_transform(o[:, None]).ravel() - te.C.values; pred = score > 0
                else: score = 1/(1 + np.exp(-o)); pred = score > 0.5
                out.append(frame(f'{arch} ({"level-regression" if obj == "R" else "direct-BCE"})', 'DL-' + obj, sd, year, te, pred, score, arch=arch))
                print(f'{mk:7s} {arch:20s} {year} {obj} seed {sd:4d} {time.time()-t0:5.1f}s', flush=True)
    pd.concat(out).to_csv(path, index=False)

In [ ]:
# ---------------- inference ----------------
import glob, itertools
from scipy import stats
B, BLOCK = 5000, 10


def bacc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    if y.min() == y.max(): return np.nan
    return 0.5*((p[y == 1] == 1).mean() + (p[y == 0] == 0).mean())


def mcc(y, p):
    tp = ((y == 1) & (p == 1)).sum(); tn = ((y == 0) & (p == 0)).sum(); fp = ((y == 0) & (p == 1)).sum(); fn = ((y == 1) & (p == 0)).sum()
    den = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
    return (tp*tn - fp*fn)/den if den > 0 else 0.0


def pt_test(y, p):
    n = len(y); py, pp = y.mean(), p.mean(); ph = (y == p).mean()
    pstar = py*pp + (1-py)*(1-pp); v1 = pstar*(1-pstar)/n
    v2 = (2*py-1)**2*pp*(1-pp)/n + (2*pp-1)**2*py*(1-py)/n + 4*py*pp*(1-py)*(1-pp)/n**2
    if v1 - v2 <= 0: return np.nan, np.nan
    s = (ph - pstar)/np.sqrt(v1 - v2); return s, 1 - stats.norm.cdf(s)


def boot_bacc(y, p, idx):
    Y, P = y[idx], p[idx]
    return 0.5*(((Y == 1) & (P == 1)).sum(1)/(Y == 1).sum(1) + ((Y == 0) & (P == 0)).sum(1)/(Y == 0).sum(1))


def signflip_p(effects):
    e = np.asarray(effects, float); e = e[~np.isnan(e)]; k = len(e); obs = abs(e.mean())
    signs = np.array(list(itertools.product([-1, 1], repeat=k)))
    return float(np.mean(np.abs((signs*e).mean(1)) >= obs - 1e-12))


def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r, i in enumerate(o):
        run = max(run, (m - r)*p[i]); adj[i] = min(1, run)
    return adj


def inference(mk, out_dir, n_feats, use_vol):
    rng = np.random.default_rng(20260926)
    pred_dir = os.path.join(out_dir, 'predictions')
    allp = pd.concat([pd.read_csv(f, parse_dates=['Date']) for f in sorted(glob.glob(os.path.join(pred_dir, 'predictions_*.csv')))], ignore_index=True)
    allp = allp[~allp.zero].copy(); allp['q'] = allp.Date.dt.year.astype(str) + 'Q' + allp.Date.dt.quarter.astype(str)
    order = ['Train-majority', 'Persistence', 'Reversal', 'MA20-trend', 'Logistic-L2', 'Ridge-return', 'Random forest', 'LightGBM']
    order += [f'{a} (level-regression)' for a in ARCHS] + [f'{a} (direct-BCE)' for a in ARCHS]

    def ensemble(m):
        g = allp[allp.method == m]
        s = g.groupby('Date').agg(y=('y', 'first'), score=('score', 'mean'), year=('year', 'first'), q=('q', 'first'))
        fam = g.family.iloc[0]
        if m == 'Train-majority': s['pred'] = g.groupby('Date').pred.first()
        elif fam == 'Naive' or m == 'Ridge-return' or fam == 'DL-R': s['pred'] = (s.score > 0).astype(int)
        else: s['pred'] = (s.score > 0.5).astype(int)
        return s.sort_index()

    ENS = {m: ensemble(m) for m in order}
    n = len(next(iter(ENS.values())))
    k = int(np.ceil(n/BLOCK)); starts = rng.integers(0, n, size=(B, k))
    IDX = ((starts[:, :, None] + np.arange(BLOCK)[None, None, :]) % n).reshape(B, -1)[:, :n]
    rows = []
    for m in order:
        g = allp[allp.method == m]
        per = g.groupby(['year', 'seed']).apply(lambda t: bacc(t.y.values, t.pred.values)).groupby('year').mean()
        e = ENS[m]; y, p = e.y.values, e.pred.values
        bb = boot_bacc(y, p, IDX); s_, ppt = pt_test(y, p)
        rows.append(dict(market=mk, method=m, family=g.family.iloc[0], **{f'bacc_{yr}': 100*v for yr, v in per.items()},
                         bacc_mean=100*per.mean(), bacc_pooled=100*bacc(y, p), ci_lo=100*np.nanpercentile(bb, 2.5),
                         ci_hi=100*np.nanpercentile(bb, 97.5), p_boot_gt50=float(np.mean(bb <= 0.5)), acc=100*(y == p).mean(),
                         mcc=mcc(y, p), up_pred=100*p.mean(), pt_stat=s_, pt_p=ppt, n=len(y)))
    TA = pd.DataFrame(rows); TA['pt_holm'] = holm(TA.pt_p.fillna(1).values)
    TA.to_csv(os.path.join(out_dir, 'tableA_skill.csv'), index=False)

    def unit_bacc(m, unit):
        g = allp[allp.method == m]
        return g.groupby([unit, 'seed']).apply(lambda t: bacc(t.y.values, t.pred.values)).groupby(unit).mean()

    def contrast(a, b, fam):
        dy = (unit_bacc(a, 'year') - unit_bacc(b, 'year')).values; dq = (unit_bacc(a, 'q') - unit_bacc(b, 'q')).values
        ea, eb = ENS[a], ENS[b]; y = ea.y.values
        dd = boot_bacc(y, ea.pred.values, IDX) - boot_bacc(y, eb.pred.values, IDX)
        obs = bacc(y, ea.pred.values) - bacc(y, eb.pred.values); se = np.nanstd(dd)
        return dict(market=mk, family=fam, A=a, B=b, delta_year_mean=100*dy.mean(), p_year=signflip_p(dy), n_q=len(dq),
                    delta_q_mean=100*dq.mean(), p_quarter=signflip_p(dq), delta_daily=100*obs, se_daily=100*se,
                    p_daily=float(2*(1 - stats.norm.cdf(abs(obs)/se))), ci_lo=100*(obs - 1.96*se), ci_hi=100*(obs + 1.96*se),
                    mde80=100*2.8*se, discordance=float((ea.pred.values != eb.pred.values).mean()))

    rowsB = [contrast(f'{a} (direct-BCE)', f'{a} (level-regression)', 'F1 objective') for a in ARCHS]
    rowsB += [contrast(f'{a} (level-regression)', 'Logistic-L2', 'F2 DL vs logistic') for a in ARCHS]
    rowsB += [contrast(f'{a} (level-regression)', 'Persistence', 'F3 DL vs persistence') for a in ARCHS]
    TB = pd.DataFrame(rowsB)
    for col in ['p_year', 'p_quarter', 'p_daily']:
        TB[col + '_holm'] = TB.groupby('family')[col].transform(lambda s: holm(s.values))
    TB.to_csv(os.path.join(out_dir, 'tableB_contrasts.csv'), index=False)
    TC = pd.DataFrame([dict(market=mk, discordance=d_, n=n, mde80_iid_pp=100*2.8*np.sqrt(d_/n)) for d_ in [0.10, 0.20, 0.30, 0.40]])
    TC.to_csv(os.path.join(out_dir, 'tableC_power.csv'), index=False)
    e0 = ENS['Logistic-L2']
    misc = dict(market=mk, volume_features=bool(use_vol), n_features=n_feats, quick=QUICK, n=int(n), up_share=float(e0.y.mean()),
                up_share_by_year={int(k_): float(v) for k_, v in e0.groupby('year').y.mean().items()},
                sessions_by_year={int(k_): int(v) for k_, v in e0.groupby('year').size().items()},
                versions=dict(numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__, lightgbm=lgb.__version__,
                              sklearn=__import__('sklearn').__version__))
    json.dump(misc, open(os.path.join(out_dir, 'misc.json'), 'w'), indent=1)
    return TA, TB, misc

In [ ]:
TUNED_MARKETS = [m for m in os.environ.get('TUNED_MARKETS', ','.join(MARKETS)).split(',') if m]
VAL_N, MAX_EP, PATIENCE = 250, (3 if QUICK else 30), (1 if QUICK else 5)
GRID = {
    'Logistic-L2 (tuned)': [dict(C=c) for c in (0.01, 0.1, 1.0)],
    'Random forest (tuned)': [dict(min_samples_leaf=m, max_depth=d) for m in (20, 50, 100) for d in (None, 5)],
    'LightGBM (tuned)': [dict(num_leaves=nl, n_estimators=ne) for nl in (7, 15) for ne in (100, 300)],
}


def make_clf(name, prm, sd):
    if name.startswith('Logistic'):
        return LogisticRegression(C=prm['C'], max_iter=5000, class_weight='balanced')
    if name.startswith('Random'):
        return RandomForestClassifier(n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=sd, class_weight='balanced', **prm)
    return lgb.LGBMClassifier(learning_rate=0.03, min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              class_weight='balanced', random_state=sd, verbose=-1, **prm)


def split_val(D, s):
    tr = D[D.Date < s].iloc[:-1]                       # outer purge
    fit, val = tr.iloc[:-(VAL_N + 1)], tr.iloc[-VAL_N:]  # inner purge of one row
    return tr, fit, val


def run_classical_tuned(D, FEATS, path, mk, log):
    if os.path.exists(path): return
    rows = []
    for year, s, e in FOLDS:
        tr, fit, val = split_val(D, s); te = D[(D.Date >= s) & (D.Date <= e)]
        if te.empty: continue
        sc_f = StandardScaler().fit(fit[FEATS]); sc = StandardScaler().fit(tr[FEATS])
        for name, grid in GRID.items():
            seeds = [0] if name.startswith('Logistic') else SEEDS
            def vscore(prm):
                v = bacc(val.y.values, make_clf(name, prm, seeds[0]).fit(sc_f.transform(fit[FEATS]), fit.y.values).predict(sc_f.transform(val[FEATS])))
                return -1 if np.isnan(v) else v
            best = max(grid, key=vscore)
            log.append(dict(market=mk, year=year, method=name, chosen=str(best)))
            for sd in seeds:
                p = make_clf(name, best, sd).fit(sc.transform(tr[FEATS]), tr.y.values).predict_proba(sc.transform(te[FEATS]))[:, 1]
                rows.append(frame(name, 'Tuned', sd, year, te, p > 0.5, p))
    pd.concat(rows).to_csv(path, index=False)


class NetT(Net):
    def __init__(s, arch, nf, h=32):
        nn.Module.__init__(s); s.arch = arch; s.drop = nn.Dropout(0.2)
        if 'LSTM' in arch: s.lstm = nn.LSTM(nf, h, batch_first=True)
        dim = h if 'LSTM' in arch else nf
        if 'CNN' in arch: s.conv = CausalConv(dim, 32); dim = 32
        if 'Attention' in arch: s.att = CausalAtt(dim)
        s.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(dim, 8), nn.ReLU(), nn.Linear(8, 1))


def train_net(arch, nf, Xtr, ytr, sd, epochs=None, Xval=None, yval=None):
    torch.manual_seed(sd); np.random.seed(sd)
    net = NetT(arch, nf); opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)
    pw = torch.tensor([(ytr == 0).sum()/max((ytr == 1).sum(), 1)], dtype=torch.float32)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr[:, None].astype(np.float32))
    best, best_ep, bad = -1, 1, 0
    for ep in range(1, (epochs or MAX_EP) + 1):
        net.train()
        for b in range(0, len(xt), 32):
            opt.zero_grad(); loss = lossf(net(xt[b:b+32]), yt[b:b+32]); loss.backward(); opt.step()
        if epochs is None:
            net.eval()
            with torch.no_grad(): pv = net(torch.from_numpy(Xval)).numpy().ravel() > 0
            sc_ = bacc(yval, pv.astype(int))
            sc_ = -1 if np.isnan(sc_) else sc_
            if sc_ > best + 1e-9: best, best_ep, bad = sc_, ep, 0
            else:
                bad += 1
                if bad >= PATIENCE: break
    return net, best_ep, best


def run_dl_tuned(D, FEATS, arch, path, mk, log):
    if os.path.exists(path): return
    X_all, W, out = D[FEATS].values, WIN[arch], []
    for year, s, e in FOLDS:
        tr_idx = np.where(D.Date < s)[0][:-1]; tr_idx = tr_idx[tr_idx >= W - 1]
        fit_idx, val_idx = tr_idx[:-(VAL_N + 1)], tr_idx[-VAL_N:]
        te_idx = np.where((D.Date >= s) & (D.Date <= e))[0]
        if len(te_idx) == 0: continue
        te = D.iloc[te_idx]; y = D.y.values
        for sd in SEEDS:
            t0 = time.time()
            sf = StandardScaler().fit(X_all[fit_idx]); Xf = np.clip(sf.transform(X_all), -10, 10)
            _, ep, vb = train_net(arch, X_all.shape[1], seqs(Xf, fit_idx, W), y[fit_idx], sd, None, seqs(Xf, val_idx, W), y[val_idx])
            st = StandardScaler().fit(X_all[tr_idx]); Xs = np.clip(st.transform(X_all), -10, 10)
            net, _, _ = train_net(arch, X_all.shape[1], seqs(Xs, tr_idx, W), y[tr_idx], sd, epochs=ep)
            net.eval()
            with torch.no_grad(): o = net(torch.from_numpy(seqs(Xs, te_idx, W))).numpy().ravel()
            p = 1/(1 + np.exp(-o))
            out.append(frame(f'{arch} (tuned)', 'Tuned', sd, year, te, p > 0.5, p, arch=arch))
            log.append(dict(market=mk, year=year, method=f'{arch} (tuned)', seed=sd, epochs=ep, val_bacc=vb))
            print(f'{mk:7s} {arch:20s} {year} tuned seed {sd:4d} ep {ep:2d} val {vb:.3f} {time.time()-t0:5.1f}s', flush=True)
    pd.concat(out).to_csv(path, index=False)


def ensemble_tuned(pred_dir):
    path = os.path.join(pred_dir, 'predictions_tuned_ensemble.csv')
    parts = [pd.read_csv(os.path.join(pred_dir, 'predictions_tuned_classical.csv'), parse_dates=['Date'])]
    parts += [pd.read_csv(os.path.join(pred_dir, f'predictions_tuned_dl_{a}.csv'), parse_dates=['Date']) for a in ARCHS]
    g = pd.concat(parts).groupby(['method', 'Date']).agg(y=('y', 'first'), zero=('zero', 'first'), year=('year', 'first'), score=('score', 'mean')).reset_index()
    e = g.groupby('Date').agg(y=('y', 'first'), zero=('zero', 'first'), year=('year', 'first'), score=('score', 'mean')).reset_index()
    e['method'], e['family'], e['seed'], e['pred'] = 'Ensemble (tuned)', 'Tuned', 0, (e.score > 0.5).astype(int)
    e[['method', 'family', 'seed', 'year', 'Date', 'y', 'zero', 'pred', 'score']].to_csv(path, index=False)


def inference_tuned(mk, out_dir):
    rng = np.random.default_rng(20260926)
    pred_dir = os.path.join(out_dir, 'predictions')
    allp = pd.concat([pd.read_csv(f, parse_dates=['Date']) for f in sorted(glob.glob(os.path.join(pred_dir, 'predictions_*.csv')))], ignore_index=True)
    allp = allp[~allp.zero].copy(); allp['q'] = allp.Date.dt.year.astype(str) + 'Q' + allp.Date.dt.quarter.astype(str)
    tuned = list(GRID) + [f'{a} (tuned)' for a in ARCHS] + ['Ensemble (tuned)']
    frozen = {'Logistic-L2 (tuned)': 'Logistic-L2', 'Random forest (tuned)': 'Random forest', 'LightGBM (tuned)': 'LightGBM'}
    frozen.update({f'{a} (tuned)': f'{a} (level-regression)' for a in ARCHS})

    def ens(m):
        g = allp[allp.method == m]
        s = g.groupby('Date').agg(y=('y', 'first'), score=('score', 'mean'), year=('year', 'first'), q=('q', 'first'))
        fam = g.family.iloc[0]
        if m == 'Train-majority': s['pred'] = g.groupby('Date').pred.first()
        elif fam == 'Naive' or m == 'Ridge-return' or fam == 'DL-R': s['pred'] = (s.score > 0).astype(int)
        else: s['pred'] = (s.score > 0.5).astype(int)
        return s.sort_index()

    names = tuned + sorted(set(frozen.values())) + ['Persistence', 'Train-majority']
    E = {m: ens(m) for m in names}
    n = len(E[tuned[0]]); k = int(np.ceil(n/BLOCK)); starts = rng.integers(0, n, size=(B, k))
    IDX = ((starts[:, :, None] + np.arange(BLOCK)[None, None, :]) % n).reshape(B, -1)[:, :n]
    rows = []
    for m in tuned:
        g = allp[allp.method == m]
        per = g.groupby(['year', 'seed']).apply(lambda t: bacc(t.y.values, t.pred.values)).groupby('year').mean()
        y, p = E[m].y.values, E[m].pred.values; bb = boot_bacc(y, p, IDX); s_, ppt = pt_test(y, p)
        rows.append(dict(market=mk, method=m, **{f'bacc_{yr}': 100*v for yr, v in per.items()}, bacc_mean=100*per.mean(),
                         bacc_pooled=100*bacc(y, p), ci_lo=100*np.nanpercentile(bb, 2.5), ci_hi=100*np.nanpercentile(bb, 97.5),
                         acc=100*(y == p).mean(), mcc=mcc(y, p), up_pred=100*p.mean(), pt_p=ppt, n=len(y)))
    TT = pd.DataFrame(rows); TT['pt_holm'] = holm(TT.pt_p.fillna(1).values)

    def ub(m, unit):
        g = allp[allp.method == m]
        return g.groupby([unit, 'seed']).apply(lambda t: bacc(t.y.values, t.pred.values)).groupby(unit).mean()

    def con(a, b, fam):
        dy = (ub(a, 'year') - ub(b, 'year')).values
        y = E[a].y.values; dd = boot_bacc(y, E[a].pred.values, IDX) - boot_bacc(y, E[b].pred.values, IDX)
        obs = bacc(y, E[a].pred.values) - bacc(y, E[b].pred.values); se = np.nanstd(dd)
        return dict(market=mk, family=fam, A=a, B=b, delta_year_mean=100*dy.mean(), p_year=signflip_p(dy), delta_daily=100*obs,
                    p_daily=float(2*(1 - stats.norm.cdf(abs(obs)/se))) if se > 0 else 1.0, mde80=100*2.8*se)

    R = [con(m, frozen[m], 'T1 tuned vs frozen') for m in frozen]
    R += [con(m, 'Logistic-L2', 'T2 tuned vs logistic') for m in tuned if m != 'Logistic-L2 (tuned)']
    R += [con(m, 'Persistence', 'T3 tuned vs persistence') for m in tuned]
    TC = pd.DataFrame(R)
    for c in ['p_year', 'p_daily']: TC[c + '_holm'] = TC.groupby('family')[c].transform(lambda s: holm(s.values))
    TT.to_csv(os.path.join(out_dir, 'tableT_tuned_skill.csv'), index=False); TC.to_csv(os.path.join(out_dir, 'tableT_tuned_contrasts.csv'), index=False)
    return TT, TC

In [ ]:
import math, random
EXT_ARCHS = [a for a in os.environ.get('EXT_ARCHS', 'LSTM,GRU,TCN,DLinear,PatchTST').split(',') if a]
ALL_EXT_ARCHS = ['LSTM', 'GRU', 'TCN', 'DLinear', 'PatchTST', 'MLP']
N_CONFIGS = 2 if QUICK else 20
EXT_MAX_EP, EXT_PATIENCE, EXT_VAL_N = (2 if QUICK else 30), (1 if QUICK else 5), 250
EXT_MARKETS = [m for m in os.environ.get('EXT_MARKETS', ','.join(MARKETS)).split(',') if m]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)
else:
    torch.set_num_threads(max(1, os.cpu_count() or 1))
print('extended arm on', DEVICE, '| configs per architecture:', N_CONFIGS)


def search_space(arch, n, seed=20261022):
    """Pre-specified random search space; drawn once, identical for every market and fold."""
    r = random.Random(seed + ALL_EXT_ARCHS.index(arch))
    cfgs = []
    for _ in range(n):
        c = dict(window=r.choice([5, 10, 20, 40, 60]), width=r.choice([16, 32, 64, 128]), layers=r.choice([1, 2]),
                 dropout=r.choice([0.0, 0.1, 0.2, 0.3]), lr=10 ** r.uniform(-4, math.log10(3e-3)),
                 wd=10 ** r.uniform(-6, -3), batch=r.choice([32, 64]))
        if arch == 'PatchTST': c['window'] = r.choice([20, 40, 60])
        cfgs.append(c)
    return cfgs


class RNNx(nn.Module):
    def __init__(s, kind, nf, c):
        super().__init__()
        rnn = nn.LSTM if kind == 'LSTM' else nn.GRU
        s.rnn = rnn(nf, c['width'], num_layers=c['layers'], batch_first=True, dropout=c['dropout'] if c['layers'] > 1 else 0.0)
        s.head = nn.Sequential(nn.Dropout(c['dropout']), nn.Linear(c['width'], 1))
    def forward(s, x): return s.head(s.rnn(x)[0][:, -1])


class TCNBlock(nn.Module):
    def __init__(s, i, o, d, p):
        super().__init__(); s.pad = 2 * d
        s.c1, s.c2 = nn.Conv1d(i, o, 3, dilation=d), nn.Conv1d(o, o, 3, dilation=d)
        s.drop = nn.Dropout(p); s.res = nn.Conv1d(i, o, 1) if i != o else nn.Identity()
    def forward(s, x):
        h = s.drop(torch.relu(s.c1(nn.functional.pad(x, (s.pad, 0)))))
        h = s.drop(torch.relu(s.c2(nn.functional.pad(h, (s.pad, 0)))))
        return torch.relu(h + s.res(x))


class TCN(nn.Module):
    """Dilated causal convolutions (dilations 1, 2, 4[, 8]); the receptive field covers windows up to 60 sessions."""
    def __init__(s, nf, c):
        super().__init__(); w = c['width']; dil = [1, 2, 4] if c['layers'] == 1 else [1, 2, 4, 8]
        s.blocks = nn.Sequential(*[TCNBlock(nf if k == 0 else w, w, d, c['dropout']) for k, d in enumerate(dil)])
        s.head = nn.Linear(w, 1)
    def forward(s, x): return s.head(s.blocks(x.transpose(1, 2))[:, :, -1])


class DLinear(nn.Module):
    """Moving-average trend/remainder decomposition with one linear map over time per component (shared across
    channels), followed by a linear read-out across channels."""
    def __init__(s, nf, c):
        super().__init__(); W = c['window']; s.k = max(3, (W // 4) | 1)
        s.lt, s.ls = nn.Linear(W, 1), nn.Linear(W, 1)
        s.head = nn.Sequential(nn.Dropout(c['dropout']), nn.Linear(nf, 1))
    def forward(s, x):
        z = x.transpose(1, 2)                                      # B, F, W
        pad = (s.k - 1) // 2
        trend = nn.functional.avg_pool1d(nn.functional.pad(z, (pad, pad), mode='replicate'), s.k, stride=1)
        return s.head((s.lt(trend) + s.ls(z - trend)).squeeze(-1))


class PatchTST(nn.Module):
    """Channel-independent patch Transformer: each feature's window is cut into patches of 8 sessions (stride 4),
    embedded, encoded by a Transformer encoder, pooled, and the per-channel outputs are combined linearly."""
    def __init__(s, nf, c):
        super().__init__(); W, d = c['window'], c['width']; s.L, s.S = 8, 4
        s.P = (W - s.L) // s.S + 1
        s.emb = nn.Linear(s.L, d); s.pos = nn.Parameter(torch.zeros(1, s.P, d))
        enc = nn.TransformerEncoderLayer(d, 4, 2 * d, c['dropout'], batch_first=True)
        s.enc = nn.TransformerEncoder(enc, c['layers'], enable_nested_tensor=False)
        s.tok = nn.Linear(d, 1); s.head = nn.Sequential(nn.Dropout(c['dropout']), nn.Linear(nf, 1))
    def forward(s, x):
        B, W, F = x.shape
        z = x.transpose(1, 2).reshape(B * F, W).unfold(1, s.L, s.S)        # B*F, P, L
        h = s.enc(s.emb(z) + s.pos).mean(1)                                  # B*F, d
        return s.head(s.tok(h).reshape(B, F))


class MLP(nn.Module):
    """Feed-forward network on the flattened input window (added at the editor's request)."""
    def __init__(s, nf, c):
        super().__init__(); d = nf * c['window']; layers = []
        for _ in range(c['layers']):
            layers += [nn.Linear(d, c['width']), nn.ReLU(), nn.Dropout(c['dropout'])]; d = c['width']
        s.net = nn.Sequential(nn.Flatten(), *layers, nn.Linear(d, 1))
    def forward(s, x): return s.net(x)


def make_ext(arch, nf, c):
    if arch in ('LSTM', 'GRU'): return RNNx(arch, nf, c)
    return {'TCN': TCN, 'DLinear': DLinear, 'PatchTST': PatchTST, 'MLP': MLP}[arch](nf, c)


def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)


def fit_ext(arch, c, Xtr, ytr, sd, epochs=None, Xval=None, yval=None):
    torch.manual_seed(sd); np.random.seed(sd); random.seed(sd)
    net = make_ext(arch, Xtr.shape[2], c).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), c['lr'], weight_decay=c['wd'])
    pw = torch.tensor([(ytr == 0).sum() / max((ytr == 1).sum(), 1)], dtype=torch.float32, device=DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    xt = torch.from_numpy(Xtr).to(DEVICE); yt = torch.from_numpy(ytr[:, None].astype(np.float32)).to(DEVICE)
    xv = torch.from_numpy(Xval).to(DEVICE) if Xval is not None else None
    best, best_ep, bad = -1.0, 1, 0
    for ep in range(1, (epochs or EXT_MAX_EP) + 1):
        net.train()
        for b in range(0, len(xt), c['batch']):            # unshuffled mini-batches, as in the other arms
            opt.zero_grad(); loss = lossf(net(xt[b:b + c['batch']]), yt[b:b + c['batch']]); loss.backward(); opt.step()
        if epochs is None:
            net.eval()
            with torch.no_grad(): pv = (net(xv) > 0).cpu().numpy().ravel().astype(int)
            v = bacc(yval, pv); v = -1.0 if np.isnan(v) else v
            if v > best + 1e-9: best, best_ep, bad = v, ep, 0
            else:
                bad += 1
                if bad >= EXT_PATIENCE: break
    return net, best_ep, best


def predict_ext(net, X):
    net.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(X), 512): out.append(net(torch.from_numpy(X[b:b + 512]).to(DEVICE)).cpu().numpy().ravel())
    return 1 / (1 + np.exp(-np.concatenate(out)))


def run_extended(D, FEATS, arch, path, mk, log_path):
    if os.path.exists(path): return
    X_all, y, out, logs = D[FEATS].values, D.y.values, [], []
    cfgs = search_space(arch, N_CONFIGS)
    for year, s, e in FOLDS:
        te_idx = np.where((D.Date >= s) & (D.Date <= e))[0]
        if len(te_idx) == 0: continue
        base_idx = np.where(D.Date < s)[0][:-1]                       # outer purge
        res = []
        for ci, c in enumerate(cfgs):
            W = c['window']; tr_idx = base_idx[base_idx >= W - 1]
            fit_idx, val_idx = tr_idx[:-(EXT_VAL_N + 1)], tr_idx[-EXT_VAL_N:]      # inner purge
            sf = StandardScaler().fit(X_all[fit_idx]); Xf = np.clip(sf.transform(X_all), -10, 10)
            t0 = time.time()
            _, ep, vb = fit_ext(arch, c, seqs(Xf, fit_idx, W), y[fit_idx], SEEDS[0], None, seqs(Xf, val_idx, W), y[val_idx])
            res.append((vb, -ci, ci, ep))
            print(f'{mk:7s} {arch:9s} {year} cfg {ci:2d} W={W:2d} width={c["width"]:3d} L={c["layers"]} '
                  f'val {vb:.3f} ep {ep:2d} {time.time() - t0:5.1f}s', flush=True)
        vb, _, ci, ep = max(res)                                        # ties -> earlier configuration
        c = cfgs[ci]; W = c['window']; tr_idx = base_idx[base_idx >= W - 1]
        st = StandardScaler().fit(X_all[tr_idx]); Xs = np.clip(st.transform(X_all), -10, 10)
        te = D.iloc[te_idx]
        for sd in SEEDS:
            net, _, _ = fit_ext(arch, c, seqs(Xs, tr_idx, W), y[tr_idx], sd, epochs=ep)
            p = predict_ext(net, seqs(Xs, te_idx, W))
            out.append(frame(f'{arch} (extended)', 'Extended', sd, year, te, p > 0.5, p, arch=arch))
        logs.append(dict(market=mk, year=year, arch=arch, config=ci, epochs=ep, val_bacc=vb, params=n_params(net),
                         n_train=len(tr_idx), **c))
    pd.concat(out).to_csv(path, index=False)
    pd.DataFrame(logs).to_csv(log_path, mode='a', index=False, header=not os.path.exists(log_path))

# Editor round: MLP baseline and stationary-feature replication

Part A adds an MLP to the extended arm on the original feature set (results/pricevol).
Part B repeats the main comparison with STATIONARY features only (results/stationary): price-level variables (raw
OHLC, close and log-volume lags and levels, moving-average levels, momentum in index points, MACD in index points)
are removed, MACD is re-expressed relative to the close, and the regression objective of the frozen networks predicts
the next-day RETURN instead of the next close level. Everything else (folds, purge, train-only scaling, seeds,
validation windows, search space, inference) is unchanged.

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss
LEVEL_VARS = (['Close_D', 'Open_D', 'High_D', 'Low_D', 'logVol_D'] + [f'close_lag{k}' for k in range(1, 6)]
              + [f'logvol_lag{k}' for k in range(1, 6)] + [f'ma_{k}' for k in (5, 10, 20, 60)]
              + [f'mom_{k}' for k in (5, 10, 20, 60)] + ['macd', 'macd_sig', 'macd_hist'])


def stationary_features(D, FEATS):
    D = D.copy()
    for c in ['macd', 'macd_sig', 'macd_hist']: D[c + '_rel'] = D[c] / D.C
    feats = [f for f in FEATS if f not in LEVEL_VARS] + ['macd_rel', 'macd_sig_rel', 'macd_hist_rel']
    return D, feats


def run_dl_stationary(D, FEATS, arch, path, mk):
    """Frozen networks with return regression (RR) instead of level regression, plus direct BCE."""
    if os.path.exists(path): return
    X_all, W, out = D[FEATS].values, WIN[arch], []
    for year, s, e in FOLDS:
        tr_idx = np.where(D.Date < s)[0][:-1]; tr_idx = tr_idx[tr_idx >= W - 1]
        te_idx = np.where((D.Date >= s) & (D.Date <= e))[0]
        if len(te_idx) == 0: continue
        sx = MinMaxScaler().fit(X_all[tr_idx]); Xs = sx.transform(X_all)
        sy = MinMaxScaler().fit(D.ret_next.values[tr_idx, None])
        Xtr, Xte = seqs(Xs, tr_idx, W), seqs(Xs, te_idx, W)
        targets = {'R': sy.transform(D.ret_next.values[tr_idx, None]).astype(np.float32), 'C': D.y.values[tr_idx, None].astype(np.float32)}
        te = D.iloc[te_idx]
        for obj in ['R', 'C']:
            for sd in SEEDS:
                torch.manual_seed(sd); np.random.seed(sd)
                net = Net(arch, X_all.shape[1]); opt = torch.optim.Adam(net.parameters(), 1e-3)
                lossf = nn.MSELoss() if obj == 'R' else nn.BCEWithLogitsLoss()
                xt, yt = torch.from_numpy(Xtr), torch.from_numpy(targets[obj])
                net.train()
                for ep in range(EPOCHS):
                    for b in range(0, len(xt), 32):
                        opt.zero_grad(); loss = lossf(net(xt[b:b+32]), yt[b:b+32]); loss.backward(); opt.step()
                net.eval()
                with torch.no_grad(): o = net(torch.from_numpy(Xte)).numpy().ravel()
                if obj == 'R': score = sy.inverse_transform(o[:, None]).ravel(); pred = score > 0
                else: score = 1/(1 + np.exp(-o)); pred = score > 0.5
                out.append(frame(f'{arch} ({"return-regression" if obj == "R" else "direct-BCE"})', 'DL-' + obj, sd, year, te, pred, score, arch=arch))
            print(f'{mk:7s} {arch:20s} {year} {obj} done', flush=True)
    pd.concat(out).to_csv(path, index=False)


def ensure_baselines(D, FEATS, pred_dir, mk):
    if not os.path.exists(os.path.join(pred_dir, 'predictions_classical.csv')):
        run_classical(D, FEATS, os.path.join(pred_dir, 'predictions_classical.csv'))
    if not os.path.exists(os.path.join(pred_dir, 'predictions_tuned_classical.csv')):
        run_classical_tuned(D, FEATS, os.path.join(pred_dir, 'predictions_tuned_classical.csv'), mk, [])

In [ ]:
# ---------------- Part A: MLP on the original features ----------------
for mk in EXT_MARKETS:
    out_dir = os.path.join(OUT_ROOT, mk); pred_dir = os.path.join(out_dir, 'predictions'); os.makedirs(pred_dir, exist_ok=True)
    A = build_aligned(mk); has_vol = A['Volume_D'].notna().all() and (A['Volume_D'] > 0).all()
    D, FEATS = build_features(A, has_vol if USE_VOLUME == 'auto' else False)
    ensure_baselines(D, FEATS, pred_dir, mk)
    run_extended(D, FEATS, 'MLP', os.path.join(pred_dir, 'predictions_ext_MLP.csv'), mk, os.path.join(OUT_ROOT, 'extended_search_log_MLP.csv'))
    print(mk, 'MLP (original features) done', flush=True)

In [ ]:
# ---------------- Part B: stationary-feature replication ----------------
OUT_S = os.path.join(ROOT, 'results', ('quick_' if QUICK else '') + 'stationary')
STAT_ARCHS = ['LSTM', 'GRU', 'TCN', 'DLinear', 'PatchTST', 'MLP']
t_s = time.time()
for mk in EXT_MARKETS:
    out_dir = os.path.join(OUT_S, mk); pred_dir = os.path.join(out_dir, 'predictions'); os.makedirs(pred_dir, exist_ok=True)
    A = build_aligned(mk); has_vol = A['Volume_D'].notna().all() and (A['Volume_D'] > 0).all()
    D0, F0 = build_features(A, has_vol if USE_VOLUME == 'auto' else False)
    D, FEATS = stationary_features(D0, F0)
    print(f'{mk}: {len(FEATS)} stationary features', flush=True)
    ensure_baselines(D, FEATS, pred_dir, mk)
    for arch in ARCHS:
        run_dl_stationary(D, FEATS, arch, os.path.join(pred_dir, f'predictions_dl_{arch}.csv'), mk)
    for arch in STAT_ARCHS:
        run_extended(D, FEATS, arch, os.path.join(pred_dir, f'predictions_ext_{arch}.csv'), mk, os.path.join(OUT_S, 'extended_search_log.csv'))
    json.dump(dict(market=mk, n_features=len(FEATS), features=FEATS), open(os.path.join(out_dir, 'features.json'), 'w'), indent=1)
    print(f'{mk} stationary replication done, elapsed {(time.time() - t_s)/60:.1f} min', flush=True)

In [ ]:
# ---------------- inference for the stationary replication (same tests as the main analysis) ----------------
Z90 = stats.norm.ppf(0.95)


def ens_scores(allp, m):
    g = allp[allp.method == m]
    s = g.groupby('Date').agg(y=('y', 'first'), score=('score', 'mean'), year=('year', 'first'), q=('q', 'first'))
    fam = g.family.iloc[0]
    if m == 'Train-majority': s['pred'] = g.groupby('Date').pred.first()
    elif fam in ('Naive', 'DL-R') or m == 'Ridge-return': s['pred'] = (s.score > 0).astype(int)
    else: s['pred'] = (s.score > 0.5).astype(int)
    return s.sort_index(), fam


def generic_inference(mk, out_dir):
    rng = np.random.default_rng(20260926)
    allp = pd.concat([pd.read_csv(f, parse_dates=['Date']) for f in sorted(glob.glob(os.path.join(out_dir, 'predictions', 'predictions_*.csv')))], ignore_index=True)
    allp = allp[~allp.zero].copy(); allp['q'] = allp.Date.dt.year.astype(str) + 'Q' + allp.Date.dt.quarter.astype(str)
    methods = list(dict.fromkeys(allp.method)); E = {m: ens_scores(allp, m) for m in methods}
    n = len(E['Logistic-L2'][0]); k = int(np.ceil(n / BLOCK)); starts = rng.integers(0, n, size=(B, k))
    IDX = ((starts[:, :, None] + np.arange(BLOCK)[None, None, :]) % n).reshape(B, -1)[:, :n]
    rows = []
    for m in methods:
        e, fam = E[m]; g = allp[allp.method == m]
        per = g.groupby(['year', 'seed']).apply(lambda t: bacc(t.y.values, t.pred.values)).groupby('year').mean()
        y, p = e.y.values, e.pred.values; bb = boot_bacc(y, p, IDX); s_, ppt = pt_test(y, p)
        auc = roc_auc_score(y, e.score.values) if m != 'Train-majority' else 0.5
        prob = fam in ('Classical', 'DL-C', 'Tuned', 'Extended') and m != 'Ridge-return'
        ll = log_loss(y, np.clip(e.score.values, 1e-6, 1 - 1e-6)) if prob else np.nan
        rows.append(dict(market=mk, method=m, family=fam, **{f'bacc_{yr}': 100*v for yr, v in per.items()}, bacc_mean=100*per.mean(),
                         bacc_pooled=100*bacc(y, p), ci_lo=100*np.nanpercentile(bb, 2.5), ci_hi=100*np.nanpercentile(bb, 97.5),
                         up_pred=100*p.mean(), auc=auc, logloss=ll, pt_p=ppt, n=len(y)))
    TS = pd.DataFrame(rows); TS['pt_holm'] = holm(TS.pt_p.fillna(1).values)

    def ub(m, unit):
        g = allp[allp.method == m]
        return g.groupby([unit, 'seed']).apply(lambda t: bacc(t.y.values, t.pred.values)).groupby(unit).mean()

    def con(a, b, fam):
        dy = (ub(a, 'year') - ub(b, 'year')).values; dq = (ub(a, 'q') - ub(b, 'q')).values
        ea, eb = E[a][0], E[b][0]; y = ea.y.values
        dd = boot_bacc(y, ea.pred.values, IDX) - boot_bacc(y, eb.pred.values, IDX)
        obs = bacc(y, ea.pred.values) - bacc(y, eb.pred.values); se = np.nanstd(dd)
        tost = max(1 - stats.norm.cdf((obs + 0.02)/se), stats.norm.cdf((obs - 0.02)/se)) if se > 0 else 1.0
        return dict(market=mk, family=fam, A=a, B=b, delta_daily=100*obs, se_daily=100*se, ci_lo=100*(obs - 1.96*se), ci_hi=100*(obs + 1.96*se),
                    p_daily=float(2*(1 - stats.norm.cdf(abs(obs)/se))) if se > 0 else 1.0, p_quarter=signflip_p(dq), p_year=signflip_p(dy),
                    mde80=100*2.8*se, min_equiv_margin=100*(abs(obs) + Z90*se), p_tost_2pp=tost)
    deep = [m for m in methods if E[m][1] in ('DL-R', 'Extended')]
    R = [con(m, 'Logistic-L2', 'S1 deep vs logistic') for m in deep] + [con(m, 'Persistence', 'S2 deep vs persistence') for m in deep]
    TC = pd.DataFrame(R)
    for c in ['p_year', 'p_quarter', 'p_daily', 'p_tost_2pp']:
        TC[c + '_holm'] = TC.groupby('family')[c].transform(lambda s: holm(s.values))
    TS.to_csv(os.path.join(out_dir, 'tableS_skill.csv'), index=False); TC.to_csv(os.path.join(out_dir, 'tableS_contrasts.csv'), index=False)
    return TS, TC


SS, SC = [], []
for mk in EXT_MARKETS:
    a, b = generic_inference(mk, os.path.join(OUT_S, mk)); SS.append(a); SC.append(b)
SS, SC = pd.concat(SS), pd.concat(SC)
SS.to_csv(os.path.join(OUT_S, 'all_markets_stationary_skill.csv'), index=False); SC.to_csv(os.path.join(OUT_S, 'all_markets_stationary_contrasts.csv'), index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print('Stationary features: four-year mean balanced accuracy (%)')
print(SS.pivot_table(index='method', columns='market', values='bacc_mean', sort=False)[EXT_MARKETS].round(2).to_string())
print('\nAUC'); print(SS.pivot_table(index='method', columns='market', values='auc', sort=False)[EXT_MARKETS].round(3).to_string())
print('\nPredicted up share (%)'); print(SS.pivot_table(index='method', columns='market', values='up_pred', sort=False)[EXT_MARKETS].round(1).to_string())
sig = SC[(SC.p_daily_holm < 0.05) | (SC.p_quarter_holm < 0.05)]
print('\nDeep vs baseline contrasts significant after Holm:'); print(sig[['market', 'family', 'A', 'delta_daily', 'p_daily_holm', 'p_quarter_holm', 'p_year_holm']].round(3).to_string() if len(sig) else 'none')

In [ ]:
# ---------------- download (no raw prices) ----------------
import zipfile as _zf
zip_path = os.path.join(ROOT, f'results_editor_round{"_quick" if QUICK else ""}.zip')   # saved in Google Drive (Run_4Markets)
with _zf.ZipFile(zip_path, 'w', _zf.ZIP_DEFLATED) as z:
    for root_, _, fs in os.walk(OUT_S):
        for f in fs: z.write(os.path.join(root_, f), os.path.relpath(os.path.join(root_, f), os.path.dirname(OUT_S)))
    for mk in EXT_MARKETS:
        f = os.path.join(OUT_ROOT, mk, 'predictions', 'predictions_ext_MLP.csv')
        if os.path.exists(f): z.write(f, os.path.relpath(f, os.path.dirname(OUT_ROOT)))
    f = os.path.join(OUT_ROOT, 'extended_search_log_MLP.csv')
    if os.path.exists(f): z.write(f, os.path.relpath(f, os.path.dirname(OUT_ROOT)))
print('created', zip_path, '(in Google Drive)' if ROOT.startswith('/content/drive') else '(local runtime only: download it now)')
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    pass